In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas_ta as ta
import math
from tqdm import tqdm
import gc
import time
import json
import pandas_ta
import optuna
optuna.logging.set_verbosity(optuna.logging.ERROR)

# You can use Matplotlib instead of Plotly for visualization by simply replacing `optuna.visualization` with
# `optuna.visualization.matplotlib` in the following examples.
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline
import talib

from sklearn.model_selection import train_test_split

from position_tools import calculate_trades, calculate_positions, count_since_last_signal

from pklibs import *

In [ ]:

# Define transaction costs and slippage
transaction_cost = 0.001  # 0.1% transaction cost per trade
slippage = 0.001  # 0.1% slippage per trade



# Define the fitness function
# def calculate_indicators(data, params):
def backtest(data, nhours, params, short_long=[-1,1], xmult=1,transaction_cost=0.001,slippage = 0.003,precision=3):
    
    # Assuming the parameters and the `data` DataFrame are correctly set up
    ema_fast_len, ema_slow_len, ema_margin_atr_len, ema_margin_atr_mult, count_since_last_ema_cross = [
        params['ema_fast_len'],
        params['ema_slow_len'],
        params['ema_margin_atr_len'],
        params['ema_margin_atr_mult'],
        params['count_since_last_ema_cross']
    ]

    # Ensure that parameters are valid
    ema_fast_len = max(1, int(ema_fast_len))
    ema_slow_len = max(1, int(ema_slow_len))
    ema_margin_atr_len = max(1, int(ema_margin_atr_len))
    count_since_last_ema_cross = int(count_since_last_ema_cross)

    # Calculate indicators using pandas_ta
    data['ema_fast'] = ta.ema(data['close'], length=ema_fast_len)
    data['ema_slow'] = ta.ema(data['close'], length=ema_slow_len)
    data['atr'] = ta.atr(data['high'], data['low'], data['close'], length=ema_margin_atr_len)
    data['ema_diff'] = data['ema_fast'] - data['ema_slow']

    # Calculate count since last EMA cross
    sign_diff = data['ema_diff'].apply(np.sign).diff().abs()
    sign_diff.fillna(0, inplace=True)

    # Convert to numpy array
    sign_diff_array = sign_diff.values

    # Apply the counting function
    data['count_since_last_ema_cross'] = count_since_last_signal(sign_diff_array)

    # Generate bull and bear signals
    bull = (
        (data['close'] > data['ema_slow']) & \
        (data['ema_diff'].abs() > (ema_margin_atr_mult * data['atr'])) & \
        # (data['count_since_last_ema_cross'] > count_since_last_ema_cross) & \
        (data['ema_fast'] > data['ema_slow']) \
    ).astype(int)

    bull_end = (
        (data['close'] < data['ema_fast']) 
    ).astype(int)

    bear = (
        (data['close'] < data['ema_slow']) & \
        (data['ema_diff'].abs() > ema_margin_atr_mult * data['atr']) & \
        # (data['count_since_last_ema_cross'] > count_since_last_ema_cross) & \
        (data['ema_fast'] < data['ema_slow']) 
    ).astype(int)

    bear_end = (
        data['close'] > data['ema_fast']
    ).astype(int)


    lclose = data['lclose'] = data.close.apply(np.log)
    
    itrades = calculate_trades(int(1 in short_long)*bull.values, int(1 in short_long)*bull_end.values, int(-1 in short_long)*bear.values, int(-1 in short_long)*bear_end.values)
    # ipositions = calculate_positions(itrades,len(lclose))
    # if not itrades.any():
    #     print('itrades:', itrades)

    entry_points = data.index[itrades[:,0]]
    exit_points = data.index[itrades[:,1]]
    trade_rets = pd.Series(itrades[:,2]*(lclose.values[itrades[:,1]] - lclose.values[itrades[:,0]]), index=data.index[itrades[:,1]]).round(precision)
    
    # trade_rets.loc[:] = trade_rets.values * (data.natr[itrades[:,0]].values)
    trade_rets *= xmult
    trade_rets -= np.log1p(transaction_cost + slippage)
    trade_pnl = trade_rets.cumsum()
    strat_pnl = pd.Series(calculate_positions(itrades, lclose.values), index=data.index)
    
    
    strat_rets = strat_pnl.diff()
    strat_rets_pct = np.expm1(strat_rets)
    trade_rets_pct = np.exp(trade_rets)
    trade_pnl_pct = np.expm1(trade_pnl)
    strat_pnl_pct = np.expm1(strat_pnl)
    
    n_trades = len(itrades)
    # strat_rets -= data['trade'].abs() * (transaction_cost + slippage)
    # data['strat_pnl'] = strat_pnl
    
    ###
    
    asset_return = lclose.iloc[-1] - lclose.iloc[0]
    # asset_return_pct = np.exp(asset_return) - 1
    strat_pnl_pct = np.expm1(strat_pnl)
    strat_rets_mean = strat_rets_pct.mean()
    # Calculate performance metrics
    if len(itrades) > 0:
        tot_return = trade_pnl.iloc[-1]
        tot_return_pct = np.expm1(tot_return)
        wins = trade_rets_pct[trade_rets > 0];  
        losses = trade_rets_pct[trade_rets < 0]
        nwins, nlosses = len(wins), len(losses)
        win_ratio = nwins/ (n_trades) if n_trades > 0 else np.nan
        profit_factor = wins.sum() / losses.abs().sum()    
        sharpe = strat_pnl_pct.iloc[-1] / strat_rets_pct.std()
        sortino = strat_rets_mean / strat_rets_pct[strat_rets_pct < 0].std()
        avg_win = wins.mean() if len(wins) > 0 else np.nan
        avg_loss = losses.mean() if len(losses) > 0 else np.nan
    else:
        tot_return, tot_return_pct = 0,0
        wins,losses,nwins,nlosses = 0,0,0,0
        win_ratio, profit_factor, sharpe, sortino, avg_win, avg_loss = np.nan,np.nan,np.nan,np.nan,np.nan,np.nan
        
    
    
    strat_running_max = strat_pnl.cummax()
    strat_drawdowns = (strat_pnl - strat_running_max)
    strat_max_drawdown = np.min(strat_drawdowns)
        
    trade_running_max = trade_pnl.cummax()
    trade_drawdowns = (trade_pnl - trade_running_max)
    trade_max_drawdown = np.min(trade_drawdowns)
    # trade_drawdowns_pct = np.exp(trade_drawdowns) - 1
    # trade_max_drawdown_pct = np.min(trade_drawdowns_pct)
    # strat_rets = 
    
    return {'tot_return':tot_return, 'tot_return_pct':tot_return_pct, 'asset_return': asset_return,
            # 'tot_return_pct':tot_return_pct, 'asset_return_pct' : asset_return_pct,
            'n_trades': n_trades, 'nwins':nwins, 'nlosses': nlosses, 'win_ratio': win_ratio, 'profit_factor': profit_factor,
            'strat_rets': strat_rets, 'strat_pnl': strat_pnl,
            'strat_rets_pct': strat_rets_pct, 'strat_pnl_pct': strat_pnl_pct,
            'trade_rets': trade_rets, 'trade_pnl': trade_pnl,
            'trade_rets_pct': trade_rets_pct, 'trade_pnl_pct': trade_pnl_pct,
            'entry_points': entry_points, 'exit_points': exit_points,
            # 
            'strat_running_max': strat_running_max, 'strat_drawdowns': strat_drawdowns, 'strat_max_drawdown': strat_max_drawdown,
            'trade_running_max': trade_running_max, 'trade_drawdowns': trade_drawdowns, 'trade_max_drawdown': trade_max_drawdown,
            # 'trade_rets_pct': trade_rets_pct, 'trade_pnl_pct': trade_pnl_pct, 'strat_rets_pct': strat_rets_pct, 'strat_pnl_pct': strat_pnl_pct, 
            'itrades': itrades,
            'avg_win': avg_win, 'avg_loss': avg_loss,
            'trade_max_drawdown': trade_max_drawdown, 'trade_drawdowns': trade_drawdowns,
            'sharpe': sharpe, 'sortino': sortino,
            # 'trade_max_drawdown_pct': trade_max_drawdown_pct, 'trade_drawdowns_pct': trade_drawdowns_pct
            # 'trade_details': trade_details 
    }
 

def optimize(data, nhours, backtest_fn = backtest, short_long=None, xmult=1,n_trials=100):
    # if short_long is None: short_long = [1,-1]
    # Define the objective function for Optuna
    def objective(trial):
        # global data, short_long, nhours, xmult
        
        
        ema_fast_len = trial.suggest_int('ema_fast_len', 17, 50)
        ema_slow_len = trial.suggest_int('ema_slow_len', ema_fast_len, 120)
        # ema_mid_len = trial.suggest_int('ema_mid_len', 20, 100)
        ema_margin_atr_len = trial.suggest_int('ema_margin_atr_len', 7, 50)
        ema_margin_atr_mult = trial.suggest_uniform('ema_margin_atr_mult', 0.1, 5)
        count_since_last_ema_cross = trial.suggest_int('count_since_last_ema_cross', 1, 20)
        # atr_filter_mult = trial.suggest_uniform('atr_filter_mult', 0.05, 2)
        # resample_len = trial.suggest_int('resample_len', 1, 40)
        
        # params = (ema_fast_len, ema_slow_len, ema_margin_atr_len, ema_margin_atr_mult)
        params = {'ema_fast_len':ema_fast_len, 'ema_slow_len':ema_slow_len, #'ema_mid_len':ema_mid_len, 
                  'ema_margin_atr_len':ema_margin_atr_len, 
                  'ema_margin_atr_mult':ema_margin_atr_mult, 
                  'count_since_last_ema_cross': count_since_last_ema_cross
                #   'atr_filter_mult': atr_filter_mult
                  }
        res = backtest_fn(data,nhours,params,short_long, xmult=xmult)
        return res['tot_return'], res['trade_max_drawdown']#/ math.cos(number_of_trades)

    # Run the optimization
    study = optuna.create_study(directions=['maximize','minimize'])
    study.optimize(objective, n_trials=n_trials)

    return study

def print_res(res):
    pairs = [
        #  performance metrics,
        (f"Total Return",		f"{100*np.expm1(res['tot_return']):.2f}%"),
        (f"Max DrawDown",		f"{100*np.expm1(res['trade_max_drawdown']):.2f}%"),
        (f"Sharpe Ratio",		f"{res['sharpe']:.2f}"),
        (f"Win Ratio",		    f"{100*res['win_ratio']:.2f}%"),
        (f"# Trades",		    f"{res['n_trades']:.0f}"),
        # (f"Asset Return",		f"{np.exp(res['asset_return'])-1:.2f}"),
        (f"# Wins",		        f"{res['nwins']:.0f}"),
        (f"# Losses",		    f"{res['nlosses']:.0f}"),
        (f"Profit Factor",		f"{res['profit_factor']:.2f}"),
        # (f"Win Ratio",		"{res['win_ratio']:.2f}"),
        # # (f"Annualized Return",		"{results['annualized_return']:.2f}"),
        # (f"Average Win",		"{res['avg_win']:.4f}"),
        # (f"Average Loss",		"{res['avg_loss']:.4f}"),
        # (f"Average DrawDown",		"{res['trade_drawdowns_pct'].mean():.4f}")
    ]
    print_table(pairs, num_columns=4)
    
def print_performance(data, nhours, asset, reset_index=False, title='', backtest_fn=backtest, params=None, btkwargs=None, window=None):
    # {'tot_return':tot_return, 'n_trades': n_trades, 'nwins':nwins, 'nlosses': nlosses, 'win_ratio': win_ratio, 'profit_factor': profit_factor, 'sharpe': sharpe, 'sortino': sortino, 'entry_points': entry_points, 'exit_points': exit_points, 'trade_rets': trade_rets, 'trade_pnl': trade_pnl}
    global res
    if reset_index:
        data = data.copy().reset_index(drop=True)
    res = backtest_fn(data,nhours,params,**btkwargs)
    print_res(res)
    itrades = res['itrades']
    
    if window is not None:
        data = data.loc[data.index[window]]
        itrades = itrades[(itrades[:,0] >= window[0]) & (itrades[:,1] <= window[1])]
    
    # Plot the results with the best parameters
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(17, 7), sharex=True, height_ratios=[1.5,1])
    # plt.figure(figsize=(14, 3))
    ax1.plot(data.index, data['close'], label='Close Price',lw=0.5)
    ax1.plot(data.index, data['ema_fast'], label='EMA Fast', color='blue',lw=0.5)
    ax1.plot(data.index, data['ema_slow'], label='SMA Slow', color='red',lw=0.5)
    # ax1.plot(data.index, data['ema_mid'], label='SMA Slow', color='gray',lw=1.5)
    ax1.fill_between(data.index, data['ema_fast'], data['ema_slow'], where=(data['ema_diff'] > 0), color='green', alpha=0.3, label='Bullish')
    ax1.fill_between(data.index, data['ema_fast'], data['ema_slow'], where=(data['ema_diff'] < 0), color='red', alpha=0.3, label='Bullish')
   
    ax1.legend(loc='best')
    plt.title('{BASE}/{QUOTE} Price and EMAs with Optimized Strategy Signals')

    ax2.axhline(0)
    ax2.plot(res['strat_pnl_pct']+1, label='Unrealized PNL', color='blue', alpha=0.7, lw=0.5)
    ax2.plot(res['trade_pnl_pct']+1, label='Cumulative PNL', color='teal', alpha=0.8, lw=2)
    # ax2.plot(data['cum_trd_rets'].apply(np.log)-1, label='Cumulative Returns', color='blue')
    # ax2.set_yscale('log',base=2)
    ax2.legend(loc='best')
    
    # for x in res['entry_points'].values: 
    #     ax1.axvline(x,color='black', lw=.5, alpha=0.2)
    #     ax2.axvline(x,color='black', lw=.5, alpha=0.2)
    # for xx in res['exit_points'].values:
    #     ax1.axvline(xx,color='black', lw=.5, alpha=0.2)
    #     ax2.axvline(xx,color='black', lw=.5, alpha=0.2)
    for i in range(len(itrades)):
        ax1.axvspan(xmin=data.index[itrades[i,0]],xmax=data.index[itrades[i,1]], color=['red','green'][(itrades[i,2]+1)//2],alpha=0.1)
        ax2.axvspan(xmin=data.index[itrades[i,0]],xmax=data.index[itrades[i,1]], color=['red','green'][(itrades[i,2]+1)//2],alpha=0.1)
    
    ax1.grid(axis='y')
    ax2.grid(axis='y')
    
    plt.show()
    return fig, res

optuna.logging.set_verbosity(optuna.logging.ERROR)
# Load the data
# asset = 'GOOGL'
# data = load_sp500_stock_candles(asset)['2010':'2025']
# data['close'] = data['adj close']
# data = load_russel2000_candles('XOXO')['2006':]
asset = 'BTC'
data = load_candles('binance',asset,'USDT','8h')#['2022':'2025']#.iloc[-35000:-5000]
nhours = 1; short_long = [1,-1]; train_ratio = .4
# data = data.resample(f'{nhours}H').agg({'open': 'first','high': 'max','low': 'min','close': 'last','volume': 'sum'})
data_train = data.iloc[:int(data.shape[0]*train_ratio)]; data_test = data.iloc[data_train.shape[0]:]
data_train, data_test = train_test_split(data, test_size=0.2, shuffle=False)
# study = optimize(data_train, nhours,backtest_fn=backtest, short_long=[1,-1],xmult=1,n_trials=500)
# aparams = [t.params for t in reversed(study.best_trials[-1:])]
# def print_performance(data, nhours, asset, reset_index=False, backtest_fn=backtest, params=None, btkwargs=None):
print(f'aparams = {aparams}')
for params in aparams:
    print('================================================================')
    print(f'params = {params}')
    # if len(data_train): 
    #     print('--- TRAIN -----------------------------------------------------')
    #     fig_tr,res_tr = print_performance(data_train,nhours,asset,reset_index=1,params=params,btkwargs={'short_long':[1], 'xmult':1}, title=f'TRAIN DATA: [0:{data_train.shape[0]}]')
    if len(data_test): 
        print('--- TEST ------------------------------------------------------')
        fig_ts, res_ts = print_performance(data_test,nhours,asset,reset_index=1,params=params,window=None,btkwargs={'short_long':[-1,1], 'xmult':1}, title=f'TEST DATA: [{data_test.shape[0]}:{data.shape[0]}]')
    


## Forward Testing

In [ ]:
# data = load_futures_candles('bybit','ADA','USDT','1h')['2020':'2023']
data = load_candles('binance','BTC','USDT','1h')['2020':'2025']
ohlc_dict = {'open': 'first','high': 'max','low': 'min','close': 'last','volume': 'sum'}
nhours = 18 ; ntrain = 3 ; ndays = 30
short_long = [1]
data = data.resample(f'{nhours}H').agg(ohlc_dict)#[:'2022']
n = data.shape[0]
w = (24 // nhours) * ndays
fwd_params = []
for i in tqdm(range(ntrain, (n // w)-1)):
    fwd_params.append(optimize(data.iloc[(i-ntrain)*w:(i)*w-1], nhours, short_long=short_long, xmult=1).best_params)
    

In [ ]:
df_params = pd.DataFrame(fwd_params)
df_params#.describe()

In [ ]:


bt_arr = []; res_arr =[]
for i in tqdm(range(ntrain, (n // w)-1)):    
    # print(i)
    wdata = indicators(data.iloc[i*w:(i+1)*w],fwd_params[i-ntrain])
    results = backtest(wdata,short_long, xmult=1)
    decorate_trades(wdata)
    bt_arr.append(wdata)
    res_arr.append(results)
    
df_res = pd.DataFrame(res_arr)
df_res.head()

In [ ]:
df_res.sharpe_ratio.mean(), (1+df_res.total_return).cumprod().values[-1], df_res.number_of_trades.sum()

In [ ]:
df_trd = pd.concat(bt_arr)
ax = df_trd['pct_strategy_returns'].cumsum().plot()
# df_trd['pct_cumulative_returns'].plot(ax=ax)

In [ ]:

# Load the data
data = load_candles('BTC','USDT','8h')['2023':'2025']
ohlc_dict = {
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
}

data = data.resample(str(21) + 'H').agg(ohlc_dict)

# data = data['2021-11-01':].copy()
short_long = [-1,1]
study = optimize(data, short_long=short_long)
best_params = study.best_params
# best_params = {'ema_fast_len': 8, 'ema_slow_len': 100, 'ema_margin_atr_len': 45, 'ema_margin_atr_mult': 2.3369417327913116}

print(f'best_params= {best_params}')
# window = data['2024-01':'2025-01'].index
window = None
print_performance(data,best_params,window=window,short_long=short_long)
print(f'Best params: {best_params}')

In [ ]:
close = np.random.uniform(2000, 3000, len(bull)).astype(int)

In [ ]:
data = load_candles(f'binance','ETH','USDT','8h')#['2021-04':]
ohlc_dict = {
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
}
test_results = []
short_long=[1]
resample_periods = np.arange(1,20,1)
for i in tqdm(resample_periods, desc="Processing"):
    rdata = data.resample(f'{i}H').agg(ohlc_dict)
    study = optimize(rdata, short_long=short_long)
    test_results.append(study)


In [ ]:
# data = load_candles('PEPE','USDT','2h')#['2021-04':]
rparams = [study.best_params for study in test_results]
rbacktests = []
short_long=[-1,1]
for i in tqdm(range(len(resample_periods))):
    rdata = data.iloc[5:].resample(f'{resample_periods[i]}H').agg(ohlc_dict)
    params = test_results[i].best_params
    rdata = indicators(rdata, params)
    test_result = backtest(rdata, short_long)
    decorate_trades(rdata)
    rbacktests.append(test_result)
    

             
# [(resample_periods[i],rbacktests[i][1]) for i in range(len(rbacktests))]

In [ ]:

suffixes = ['_suffix', '_suffix2']

# Function to remove suffixes from a key
def remove_suffixes(key, suffixes):
    for suffix in suffixes:
        if key.endswith(suffix):
            return key[:-len(suffix)]
    return key

# Function to remove the suffixes from dictionary keys
def remove_suffixes_from_dict(d, suffixes):
    return {remove_suffixes(key, suffixes): value for key, value in d.items()}

dfres = pd.DataFrame([{'period': i, 'days':int(i*2/24),  **(remove_suffixes_from_dict(d1.best_params, ['_len'])), **d2} for i, d1, d2 in zip(resample_periods,test_results, rbacktests)])

dfres.head(10)

dfres.sort_values(by='total_return', axis=0, ascending=False).head(10)

In [ ]:

dfres.sort_values(by='total_return', axis=0, ascending=False).head(50)

In [ ]:
# i = 81
data = load_candles('binance','ETH','USDT','8h')#['2022':]
i=16
rdata = data.resample(f'{resample_periods[i]}H').agg(ohlc_dict)#['2021-04':]
# study = test_results[i]
short_long = [1,-1]
study = optimize(rdata, short_long=short_long)
params = study.best_params
window = None
# window = data['2023-06':'2025-01'].index
print_performance(rdata, test_results[i].best_params, short_long=short_long)

print(params)

In [ ]:

data = load_candles('ETC','USDT','1d')#['2021-04':]
data.join(data.resample(f'{12}H').agg(ohlc_dict),rsuffix="_res")